# Regex Basics in SQL

## Overview
Regular expressions (regex) are patterns for matching text. In SQL databases they extend the simple wildcard matching of `LIKE` to handle complex, real-world data patterns in healthcare records: phone numbers in multiple formats, medical record numbers, ICD codes, lab result strings, and free-text clinical notes.

**SQLite note:** SQLite does not have a built-in `REGEXP` operator, but registers one via Python's `re` module in these notebooks. PostgreSQL has native POSIX regex operators built in.

**Pattern operator comparison:**

| Need | SQLite (these notebooks) | PostgreSQL |
|---|---|---|
| Match a pattern | `col REGEXP 'pattern'` | `col ~ 'pattern'` |
| Case-insensitive | Built into registered function | `col ~* 'pattern'` |
| Does not match | `NOT (col REGEXP 'pattern')` | `col !~ 'pattern'` |
| Extract match | Python `re.search()` | `regexp_match(col, 'pattern')` |
| Replace | `regexp_replace(col, pat, repl)` | `regexp_replace(col, pat, repl)` |

**PostgreSQL regex flavour:** POSIX ERE (Extended Regular Expressions) — same as Python `re` without look-around.

---

In [1]:
import sqlite3, re

def make_conn():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    # Register Python re as SQLite's REGEXP operator
    conn.create_function("regexp", 2,
        lambda pattern, string: bool(re.search(pattern, string or "", re.IGNORECASE)))
    conn.create_function("regexp_cs", 2,
        lambda pattern, string: bool(re.search(pattern, string or "")))
    conn.create_function("regexp_replace", 3,
        lambda pattern, string, repl: re.sub(pattern, repl, string or "", flags=re.IGNORECASE))
    conn.create_function("regexp_replace_cs", 3,
        lambda pattern, string, repl: re.sub(pattern, repl, string or ""))
    return conn

conn = make_conn()
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    dob          TEXT,
    phone        TEXT,
    email        TEXT,
    province     TEXT,
    mrn          TEXT,          -- medical record number: format MRN-XXXXXX
    postal_code  TEXT
);
CREATE TABLE clinical_notes (
    note_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id   TEXT NOT NULL,
    note_date    TEXT NOT NULL,
    note_text    TEXT NOT NULL,
    note_type    TEXT NOT NULL DEFAULT 'progress'
);
CREATE TABLE medications (
    med_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id   TEXT NOT NULL,
    drug_name    TEXT NOT NULL,
    dosage       TEXT,          -- free text: '10mg', '2.5 mg', '500 MG', '10mg/day'
    frequency    TEXT           -- free text: 'twice daily', 'BID', 'q12h', '2x/day'
);
CREATE TABLE lab_results (
    lab_id       INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id   TEXT NOT NULL,
    test_name    TEXT NOT NULL,
    raw_value    TEXT NOT NULL,  -- messy: '7.2 mmol/L', '7.2mmol/l', '<5', '>10', '~8'
    result_date  TEXT NOT NULL
);
INSERT INTO patients VALUES
    ('P001','Aroha Ngata',        '1985-03-15','506-555-0101','aroha@email.com','NB','MRN-004521','E2C 1A1'),
    ('P002','Liam Chen',          '1990-07-22','(416) 555-2233','liam.chen@gmail.com','ON','MRN-008823','M5V 3A1'),
    ('P003','Fatima Al-Rashid',   '1978-11-05','604.555.9900','f.rashid@hotmail.com','BC','MRN-012234','V6B 2W9'),
    ('P004','James MacLeod',      '2001-01-30','902 555 7788','jmacleod@yahoo.ca','NS','MRN-003344','B3J 1R2'),
    ('P005','Mei-Ling Tran',      '1965-08-19','514-555-1122','mei.tran@email.com','QC','MRN-019900','H2Y 1C6'),
    ('P006','invalid_email_user', '1970-01-01','not-a-phone','bad@@email','ON',NULL,'not_postal');
INSERT INTO clinical_notes VALUES
    (NULL,'P001','2023-04-10','Patient presents with BP 142/88 mmHg. Prescribed Lisinopril 10mg daily. Follow-up in 6 weeks.','progress'),
    (NULL,'P001','2023-10-01','BP improved to 128/82 mmHg. HbA1c 7.2%. Continued Lisinopril. Added Metformin 500mg BID.','progress'),
    (NULL,'P002','2023-05-15','Asthma exacerbation. SpO2 94%. Salbutamol 2.5mg nebulised. Prescribed Fluticasone 250mcg/dose.','urgent'),
    (NULL,'P003','2023-08-20','Obesity BMI 34.2. Referred to dietitian. BP 148/92 mmHg. Consider ACE inhibitor.','referral'),
    (NULL,'P004','2023-09-01','Healthy adult. No complaints. BP 118/76 mmHg. ECG normal sinus rhythm.','routine'),
    (NULL,'P005','2023-11-10','HbA1c 8.9% — poorly controlled T2DM. Insulin glargine 20 units nocte. Metformin 1g BD.','progress');
INSERT INTO medications VALUES
    (NULL,'P001','Lisinopril','10mg','once daily'),
    (NULL,'P001','Metformin','500mg','BID'),
    (NULL,'P002','Salbutamol','2.5mg','PRN'),
    (NULL,'P002','Fluticasone','250mcg','BD'),
    (NULL,'P003','Amlodipine','5 mg','once daily'),
    (NULL,'P005','Metformin','1g','BD'),
    (NULL,'P005','Insulin glargine','20 units','nocte'),
    (NULL,'P001','Atorvastatin','40MG','once daily'),
    (NULL,'P003','Ramipril','2.5 mg/day','once daily');
INSERT INTO lab_results VALUES
    (NULL,'P001','HbA1c','7.2%','2023-10-01'),
    (NULL,'P001','eGFR','>60 mL/min/1.73m2','2023-10-01'),
    (NULL,'P002','SpO2','94%','2023-05-15'),
    (NULL,'P003','BMI','34.2 kg/m2','2023-08-20'),
    (NULL,'P005','HbA1c','8.9 %','2023-11-10'),
    (NULL,'P005','HbA1c','<5.7%','2022-01-01'),
    (NULL,'P004','BP','118/76 mmHg','2023-09-01'),
    (NULL,'P001','BP','142/88 mmHg','2023-04-10'),
    (NULL,'P001','BP','128/82 mmHg','2023-10-01');
""")
conn.commit()
print("Healthcare dataset ready")
for tbl in ("patients","clinical_notes","medications","lab_results"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")


Healthcare dataset ready
  patients: 6 rows
  clinical_notes: 6 rows
  medications: 9 rows
  lab_results: 9 rows


---
## Operators and phone format matching

In [2]:
print("=== Regex operators: LIKE vs SQLite REGEXP vs PostgreSQL ~ ===")
print()

# SQLite: uses REGEXP (backed by Python re via registered function)
# PostgreSQL: ~  (case-sensitive match)
#             ~* (case-insensitive match)
#             !~ / !~* (negation)

operator_table = [
    ("LIKE 'A%'",        "Starts with A",               "No regex; only % and _ wildcards"),
    ("LIKE '%NB%'",      "Contains NB",                 "Case-sensitive in most DBs; use ILIKE in PostgreSQL for insensitive"),
    ("SIMILAR TO",       "POSIX-like syntax via SQL std","Rarely used; ~ is preferred in PostgreSQL"),
    ("~ 'pattern'",      "Case-sensitive regex match",  "PostgreSQL; returns boolean"),
    ("~* 'pattern'",     "Case-insensitive regex match","PostgreSQL; most common"),
    ("!~ 'pattern'",     "Does NOT match (case-sensitive)","Negation of ~"),
    ("!~* 'pattern'",    "Does NOT match (case-insensitive)","Negation of ~*"),
    ("REGEXP 'pattern'", "SQLite (with custom function)","Registered via Python re"),
]
print(f"  {'Operator':<28s}  {'Meaning':<35s}  Notes")
print("  " + "-"*80)
for op, meaning, note in operator_table:
    print(f"  {op:<28s}  {meaning:<35s}  {note}")


=== Regex operators: LIKE vs SQLite REGEXP vs PostgreSQL ~ ===

  Operator                      Meaning                              Notes
  --------------------------------------------------------------------------------
  LIKE 'A%'                     Starts with A                        No regex; only % and _ wildcards
  LIKE '%NB%'                   Contains NB                          Case-sensitive in most DBs; use ILIKE in PostgreSQL for insensitive
  SIMILAR TO                    POSIX-like syntax via SQL std        Rarely used; ~ is preferred in PostgreSQL
  ~ 'pattern'                   Case-sensitive regex match           PostgreSQL; returns boolean
  ~* 'pattern'                  Case-insensitive regex match         PostgreSQL; most common
  !~ 'pattern'                  Does NOT match (case-sensitive)      Negation of ~
  !~* 'pattern'                 Does NOT match (case-insensitive)    Negation of ~*
  REGEXP 'pattern'              SQLite (with custom function)        Re

In [3]:
# ── Live demos: find patients by phone format ────────────────────
print("=== Phone number format matching ===")

phones = conn.execute("SELECT patient_id, full_name, phone FROM patients").fetchall()
print("All phone values in the table:")
for r in phones:
    print(f"  {r['patient_id']}  {r['phone']!r}")

print()
# Format: 506-555-0101 (dashes)
print("Dash format (NNN-NNN-NNNN):")
rows = conn.execute("""
    SELECT patient_id, phone FROM patients
    WHERE phone REGEXP '^[0-9]{3}-[0-9]{3}-[0-9]{4}$'
""").fetchall()
for r in rows: print(f"  {r['patient_id']}  {r['phone']}")

# Format: (416) 555-2233
print("Parentheses format:")
rows = conn.execute(r"""
    SELECT patient_id, phone FROM patients
    WHERE phone REGEXP '^\([0-9]{3}\) [0-9]{3}-[0-9]{4}$'
""").fetchall()
for r in rows: print(f"  {r['patient_id']}  {r['phone']}")

# Any valid Canadian phone (multiple formats)
print("Any valid phone (3 digits + separator + 3 + separator + 4):")
rows = conn.execute(r"""
    SELECT patient_id, phone FROM patients
    WHERE phone REGEXP '^[\(]?[0-9]{3}[\)\-\. ]? ?[0-9]{3}[\-\. ][0-9]{4}$'
""").fetchall()
for r in rows: print(f"  {r['patient_id']}  {r['phone']}")

print()
print("PostgreSQL equivalents:")
print("  WHERE phone ~  '^[0-9]{3}-[0-9]{3}-[0-9]{4}$'     -- case-sensitive")
print("  WHERE phone ~* '^[0-9]{3}-[0-9]{3}-[0-9]{4}$'     -- case-insensitive (same for digits)")


=== Phone number format matching ===
All phone values in the table:
  P001  '506-555-0101'
  P002  '(416) 555-2233'
  P003  '604.555.9900'
  P004  '902 555 7788'
  P005  '514-555-1122'
  P006  'not-a-phone'

Dash format (NNN-NNN-NNNN):
  P001  506-555-0101
  P005  514-555-1122
Parentheses format:
  P002  (416) 555-2233
Any valid phone (3 digits + separator + 3 + separator + 4):
  P001  506-555-0101
  P002  (416) 555-2233
  P003  604.555.9900
  P004  902 555 7788
  P005  514-555-1122

PostgreSQL equivalents:
  WHERE phone ~  '^[0-9]{3}-[0-9]{3}-[0-9]{4}$'     -- case-sensitive
  WHERE phone ~* '^[0-9]{3}-[0-9]{3}-[0-9]{4}$'     -- case-insensitive (same for digits)


---
## Character classes, quantifiers, and anchors

In [4]:
print("=== Character classes and quantifiers ===")

char_classes = [
    ("[abc]",      "Any one of a, b, c"),
    ("[a-z]",      "Any lowercase letter"),
    ("[A-Z]",      "Any uppercase letter"),
    ("[0-9]",      "Any digit (same as \\d in POSIX)"),
    ("[^abc]",     "NOT a, b, or c"),
    ("[[:alpha:]]","POSIX: any letter (PostgreSQL syntax)"),
    ("[[:digit:]]","POSIX: any digit (PostgreSQL syntax)"),
    ("[[:alnum:]]","POSIX: any letter or digit"),
    ("[[:space:]]","POSIX: any whitespace"),
    (".",          "Any single character except newline"),
    ("\\s",         "Whitespace (space, tab, newline)"),
    ("\\d",         "Digit [0-9]"),
    ("\\w",         "Word character [a-zA-Z0-9_]"),
    ("\\b",         "Word boundary"),
]
quantifiers = [
    ("*",    "0 or more"),
    ("+",    "1 or more"),
    ("?",    "0 or 1 (optional)"),
    ("{n}",  "Exactly n"),
    ("{n,}", "n or more"),
    ("{n,m}","Between n and m"),
]
anchors = [
    ("^",  "Start of string"),
    ("$",  "End of string"),
    ("|",  "Alternation (OR): cat|dog"),
    ("()", "Capture group"),
    ("(?:)","Non-capturing group (PostgreSQL 15+, not SQLite)"),
]

print("Character classes:")
for cls, desc in char_classes:
    print(f"  {cls:<16s}  {desc}")
print()
print("Quantifiers:")
for q, desc in quantifiers:
    print(f"  {q:<8s}  {desc}")
print()
print("Anchors and grouping:")
for a, desc in anchors:
    print(f"  {a:<8s}  {desc}")

print()
print("=== Validating MRN format: MRN-XXXXXX (6 digits) ===")
rows = conn.execute(r"""
    SELECT patient_id, full_name, mrn,
           CASE WHEN mrn REGEXP '^MRN-[0-9]{6}$' THEN 'valid' ELSE 'invalid/null' END AS mrn_status
    FROM patients
    ORDER BY patient_id
""").fetchall()
for r in rows:
    print(f"  {r['patient_id']}  {r['full_name']:<20s}  mrn={r['mrn']!r:<14s}  {r['mrn_status']}")


=== Character classes and quantifiers ===
Character classes:
  [abc]             Any one of a, b, c
  [a-z]             Any lowercase letter
  [A-Z]             Any uppercase letter
  [0-9]             Any digit (same as \d in POSIX)
  [^abc]            NOT a, b, or c
  [[:alpha:]]       POSIX: any letter (PostgreSQL syntax)
  [[:digit:]]       POSIX: any digit (PostgreSQL syntax)
  [[:alnum:]]       POSIX: any letter or digit
  [[:space:]]       POSIX: any whitespace
  .                 Any single character except newline
  \s                Whitespace (space, tab, newline)
  \d                Digit [0-9]
  \w                Word character [a-zA-Z0-9_]
  \b                Word boundary

Quantifiers:
  *         0 or more
  +         1 or more
  ?         0 or 1 (optional)
  {n}       Exactly n
  {n,}      n or more
  {n,m}     Between n and m

Anchors and grouping:
  ^         Start of string
  $         End of string
  |         Alternation (OR): cat|dog
  ()        Capture group
  (

---
## Email and postal code validation

In [5]:
print("=== Email and postal code validation ===")

# Email validation (practical, not RFC-5321 complete)
print("Email validation:")
rows = conn.execute(r"""
    SELECT patient_id, email,
           CASE WHEN email REGEXP '^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
                THEN 'valid' ELSE 'invalid' END AS email_status
    FROM patients
    ORDER BY patient_id
""").fetchall()
for r in rows:
    print(f"  {r['patient_id']}  {r['email']!r:<30s}  {r['email_status']}")

# Canadian postal code: A1A 1A1 (letter-digit-letter space digit-letter-digit)
print()
print("Canadian postal code validation:")
rows = conn.execute(r"""
    SELECT patient_id, postal_code,
           CASE WHEN postal_code REGEXP '^[A-Za-z][0-9][A-Za-z] [0-9][A-Za-z][0-9]$'
                THEN 'valid' ELSE 'invalid' END AS postal_status
    FROM patients
    ORDER BY patient_id
""").fetchall()
for r in rows:
    print(f"  {r['patient_id']}  {r['postal_code']!r:<12s}  {r['postal_status']}")

print()
print("PostgreSQL equivalents:")
pg = [
    ("Email check",   "WHERE email ~ '^[a-zA-Z0-9._%+\\-]+@[a-zA-Z0-9.\\-]+\\.[a-zA-Z]{2,}$'"),
    ("Postal code",   "WHERE postal_code ~* '^[A-Z][0-9][A-Z] [0-9][A-Z][0-9]$'"),
    ("MRN format",    "WHERE mrn ~ '^MRN-[0-9]{6}$'"),
    ("Find invalids", "WHERE email !~ '^[a-zA-Z0-9._%+\\-]+@[a-zA-Z0-9.\\-]+\\.[a-zA-Z]{2,}$'"),
]
for label, sql in pg:
    print(f"  {label}: {sql}")


=== Email and postal code validation ===
Email validation:
  P001  'aroha@email.com'               valid
  P002  'liam.chen@gmail.com'           valid
  P003  'f.rashid@hotmail.com'          valid
  P004  'jmacleod@yahoo.ca'             valid
  P005  'mei.tran@email.com'            valid
  P006  'bad@@email'                    invalid

Canadian postal code validation:
  P001  'E2C 1A1'     valid
  P002  'M5V 3A1'     valid
  P003  'V6B 2W9'     valid
  P004  'B3J 1R2'     valid
  P005  'H2Y 1C6'     valid
  P006  'not_postal'  invalid

PostgreSQL equivalents:
  Email check: WHERE email ~ '^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
  Postal code: WHERE postal_code ~* '^[A-Z][0-9][A-Z] [0-9][A-Z][0-9]$'
  MRN format: WHERE mrn ~ '^MRN-[0-9]{6}$'
  Find invalids: WHERE email !~ '^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'


---
## Common Pitfalls

**1. Confusing LIKE wildcards with regex metacharacters**
`LIKE '%bp%'` uses SQL wildcard `%` (any string). In regex, `.` means any character and `*` means zero or more of the preceding. `WHERE note_text ~ 'bp*'` means 'b followed by zero or more p characters' -- not 'anything containing bp'. LIKE and regex are completely different pattern languages. Never mix their syntax.

**2. Missing anchors causing unintended partial matches**
`WHERE phone ~ '[0-9]{10}'` matches any string containing 10 consecutive digits, including `phone123456789012345`. Add anchors: `~ '^[0-9]{10}$'` to match the whole string. Without `^` and `$`, regex always does a substring search.

**3. Case sensitivity surprises**
PostgreSQL `~` is case-sensitive: `~ 'mrn'` does not match `'MRN-004521'`. Use `~*` for case-insensitive matching. SQLite's custom `REGEXP` function behaviour depends on flags passed to `re.search()`. In these notebooks the function uses `re.IGNORECASE`.

**4. Backslash escaping in SQL string literals**
In PostgreSQL, `E'\\d+'` uses the escape string syntax for a literal `\d+` regex. In standard SQL strings, `'\\d+'` may require doubling: `'\\\\d+'`. Use dollar-quoting in PostgreSQL to avoid confusion: `$$\d+$$`. In Python, use raw strings: `r'\d+'`.

**5. Using regex for simple prefix/suffix checks instead of LIKE**
`WHERE name ~ '^Aroha'` works but is slower than `WHERE name LIKE 'Aroha%'` because LIKE with a constant prefix can use a B-tree index, while regex generally cannot. Use LIKE for simple prefix/suffix patterns; reserve regex for patterns that genuinely require it.

**6. NULL values not matching any regex**
`WHERE phone ~ '^[0-9]'` returns FALSE (not TRUE) for NULL phone values -- NULLs are excluded silently. If you want to include NULLs in a 'not valid' category, use `WHERE phone IS NULL OR phone !~ '^[0-9]{3}'`.


---
*sql_methods_library - Samantha McGarrigle*